# Jour 2 · Chapitre 4 — Scénario : nettoyer le fichier contrats avant l'audit interne
## Formation Beobank · SAS → Python · Cahier participant

**L'histoire.** L'audit interne arrive vendredi. Thomas, contrôleur de gestion, vous confie
`CTR.csv` :
*"L'auditeur va recalculer nos indicateurs. Je veux un fichier propre : types corrects, trous
traités, doublons supprimés, valeurs aberrantes signalées. Et un graphique pour la réunion."*

**Comment travailler.** Remplacez chaque `______`. Le scénario suit l'ordre du cours : Series et
DataFrames, lecture, analyse, nettoyage (cellules vides, mauvais format, données fausses,
doublons), corrélations, graphiques.

📎 **Matière à relire :** `Cours_Pandas.ipynb` (même dossier) — sections 3 à 13.

## Préparation (déjà écrite, exécutez-la)

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────
from io import StringIO
import matplotlib.pyplot as plt
import pandas as pd

DATA = "../../data/"
PARAMS = dict(sep=";", na_values=".", encoding="utf-8")
print("Prêt.")

---
## Étape 1 — Series et DataFrame : le vocabulaire de l'auditeur

*"L'auditeur m'a envoyé ses 3 contrôles sous forme de petit tableau, tu peux le recréer ?"*

In [ ]:
# ── Créer une Series et un DataFrame à la main ────────────────────────────
# Quel objet pandas représente UNE colonne étiquetée ?
seuils = pd.______([0, -10000, 100000], index=["min_solde", "alerte", "max_solde"])
print(seuils["alerte"])                              # → -10000
# Quel objet pandas représente un TABLEAU à plusieurs colonnes ?
controles = pd.______({"controle": ["manquants", "doublons", "aberrants"],
                             "statut":   ["à faire", "à faire", "à faire"]})
controles

---
## Étape 2 — Charger et diagnostiquer

In [ ]:
# ── Lecture et analyse ────────────────────────────────────────────────────
ctr = pd.read_csv(DATA + "CTR.csv", **PARAMS)
# Quel attribut donne (lignes, colonnes) ?
print(ctr.______)                                    # → (200, 11)
# Quelle méthode affiche types et nombre de valeurs non nulles par colonne ?
ctr.______()

In [ ]:
# Quelle méthode résume les colonnes numériques (count, mean, std, min, max...) ?
ctr[["SLD_CTR", "SLD_DSP", "MNT_INI"]].______().round(2)

---
## Étape 3 — Les cellules vides

*"Si le solde disponible manque, prends le solde comptable. Et sors-moi un fichier des seuls
contrats dont le solde est connu."*

In [ ]:
# ── Traiter les valeurs manquantes ────────────────────────────────────────
# Quelle méthode repère les cases vides ? Laquelle additionne les True ?
print("SLD_DSP manquants avant :", ctr["SLD_DSP"].______().______())       # → 153
# Quelle méthode remplit les cases vides ? Avec quelle COLONNE les remplir ?
ctr["SLD_DSP"] = ctr["SLD_DSP"].______(ctr["______"])
print("SLD_DSP manquants après :", ctr["SLD_DSP"].isna().sum())
# Quelle méthode supprime les lignes vides ? Quel paramètre limite la vérification à certaines colonnes ?
ctr_soldes = ctr.______(______=["SLD_CTR"])
print("Contrats avec solde :", len(ctr_soldes))                       # → 63

---
## Étape 4 — Le mauvais format

*"Les dates sont du texte, l'auditeur ne pourra pas calculer d'ancienneté."*

In [ ]:
# ── Convertir les dates ───────────────────────────────────────────────────
for col in ["DAT_OUV_CTR", "DAT_CLO_CTR"]:
    # Quelle valeur de errors= remplace une date illisible par NaT au lieu de planter ?
    ctr[col] = pd.to_datetime(ctr[col], errors="______")
print(ctr[["DAT_OUV_CTR", "DAT_CLO_CTR"]].dtypes)

---
## Étape 5 — Les données fausses

*"Tout solde sous -100 000 € doit être vérifié à la main : marque-les d'un drapeau."*

In [ ]:
# ── Signaler les valeurs aberrantes ───────────────────────────────────────
ctr["A_VERIFIER"] = False
# Quel accesseur permet de sélectionner des lignes par condition ET une colonne, pour y écrire ?
ctr.______[ctr["SLD_CTR"] < -100000, "A_VERIFIER"] = True
# Comment compter les True d'une colonne booléenne ?
print("Contrats à vérifier :", ctr["A_VERIFIER"].______())
ctr.loc[ctr["A_VERIFIER"], ["IDT_AC", "SLD_CTR"]]

---
## Étape 6 — Les doublons

*"Un ancien export a recopié 3 lignes deux fois. Vérifie que ton fichier final n'en a pas."*

In [ ]:
# ── Détecter et supprimer les doublons ────────────────────────────────────
export_sale = pd.concat([ctr, ctr.head(3)])          # simulation de l'ancien export
# Quelle méthode marque les lignes déjà vues ?
print("Doublons :", export_sale.______().sum())                 # → 3
# Quelle méthode les supprime ?
export_propre = export_sale.______()
print("Lignes après nettoyage :", len(export_propre))                 # → 200

---
## Étape 7 — Les corrélations

*"Le solde disponible suit-il le solde comptable ?"*

In [ ]:
# ── Matrice de corrélation ────────────────────────────────────────────────
# Quelle méthode calcule la corrélation entre toutes les colonnes numériques ?
ctr[["SLD_CTR", "SLD_DSP", "MNT_INI"]].______().round(2)

---
## Étape 8 — Envoyer un extrait en JSON à l'outil d'audit

In [ ]:
# ── Écrire et relire du JSON ──────────────────────────────────────────────
extrait = ctr[["IDT_AC", "COD_DEV", "SLD_CTR"]].head(2)
# Quelle méthode convertit un DataFrame en texte JSON ?
texte_json = extrait.______(orient="records")
print(texte_json)
# Quelle fonction pandas relit du JSON ? (StringIO fait passer le texte pour un fichier)
pd.______(StringIO(texte_json))

---
## 🏁 Étape finale — Le graphique de la réunion

*"Un diagramme en barres : nombre de contrats par devise."*

In [ ]:
# ── Graphique ─────────────────────────────────────────────────────────────
# Quel type de graphique (kind=) dessine des barres verticales ?
ax = ctr["COD_DEV"].value_counts().plot(kind="______", color="#2E5A88")
# Quelle méthode de l'axe ajoute un titre ?
ax.______("Contrats par devise")
ax.set_ylabel("Nombre de contrats")
plt.show()

✍️ **Note pour l'auditeur (3 points) :** résumez ce que vous avez corrigé et ce qui reste à
vérifier à la main.